In [ ]:
import pandas as pd
import re
from data_gatherer.data_gatherer import DataGatherer
from data_gatherer.parser.xml_parser import XMLParser

In [ ]:
df = pd.read_parquet("scripts/exp_input/Local_fetched_data.parquet")

dg = DataGatherer(log_level='INFO')

if dg.parser is None:
    dg.parser = XMLParser(dg.open_data_repos_ontology, dg.logger, llm_name=dg.llm)

In [ ]:
# there are some pcm ids in this csv article_ids_REV_test.csv that we want to filter the df on
article_ids = pd.read_csv("scripts/exp_input/REV_test.txt", header=None, names=['publication'])['publication'].tolist()
article_ids = [re.sub(r'https://www.ncbi.nlm.nih.gov/pmc/articles/', '', id.lower()) for id in article_ids]
len(article_ids), len(df)

In [ ]:
df_filtered = df[df['publication'].str.lower().isin([id.lower() for id in article_ids])]
df_filtered['format'].value_counts()

In [ ]:
# # Flan-t5-finetuned -- base
# ! bash k8s/run_loop.sh \
# --iterations 1 \
# --gpus 1 \
# --input article_ids_REV_test.csv \
# --max-articles-per-slice 249 \
# --output-dir k8s/output/rev_test_c1 \
# --seed-ontology data_gatherer/config/open_bio_data_repos.json \
# --job-suffix -tc1 \
# --semantic-retrieval false \
# --brute-force-regex false \
# --no-enrich

In [ ]:
# # Flan-t5-finetuned -- S3
# ! bash k8s/run_loop.sh \
#     --iterations 1 --gpus 1 \
#     --input article_ids_REV_test.csv \
#     --max-articles-per-slice 249 \
#     --output-dir k8s/output/rev_test_c2 \
#     --seed-ontology data_gatherer/config/open_bio_data_repos.json \
#     --job-suffix -tc2 \
#     --semantic-retrieval true \
#     --top-k 3 \
#     --brute-force-regex false \
#     --no-enrich

In [ ]:
# # Flan-t5-finetuned -- RS3
# ! bash k8s/run_loop.sh \
#     --iterations 1 --gpus 1 \
#     --input article_ids_REV_test.csv \
#     --max-articles-per-slice 249 \
#     --output-dir k8s/output/rev_test_c3 \
#     --seed-ontology data_gatherer/config/open_bio_data_repos.json \
#     --job-suffix -tc3 \
#     --semantic-retrieval true \
#     --top-k 3 \
#     --brute-force-regex true \
#     --no-enrich

In [ ]:
# # Flan-t5-finetuned -- Full Document Chunk
# ! bash k8s/run_loop.sh \
#       --iterations 1 --gpus 1 \
#       --input article_ids_REV_test.csv \
#       --max-articles-per-slice 249 \
#       --output-dir k8s/output/rev_test_c4 \
#       --seed-ontology data_gatherer/config/open_bio_data_repos.json \
#       --job-suffix -tc4 \
#       --top-k all \
#       --semantic-retrieval false \
#       --brute-force-regex false \
#       --no-enrich

In [ ]:
# # Claude Haiku 4.5 -- base 
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c1 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --semantic-retrieval false --brute-force-regex false \
#     --use-batch-api false

In [ ]:
# # Claude Haiku 4.5 -- S3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c2 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex false \
#     --use-batch-api false

In [ ]:
# # Claude Haiku 4.5 -- RS3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c3 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex true \
#     --use-batch-api false

In [ ]:
# # Claude Haiku 4.5 — FDR
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c4 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --full-document-read true \
#     --prompt-name CLAUDE_FDR_FewShot

In [ ]:
# batch_id = 'msgbatch_012T8HnpBDzNrSMhfAt2VGVL'

# res = dg.parser.llm_client.download_batch_results(
#     batch_id=batch_id,
#     output_file_path='scripts/tmp/resp_RTR_base.jsonl',
#     api_provider='anthropic',
# )

# res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_haiku_c4/dataset_citations.csv')

In [ ]:
# # gpt 5 mini - Base
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c1 \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval false --brute-force-regex false

In [ ]:
# batch_id = 'batch_6a46dc3f24a48190ae58aba354e72101'

# res = dg.parser.llm_client.download_batch_results(
#     batch_id=batch_id,
#     output_file_path='scripts/tmp/resp_RTR_base.jsonl',
#     api_provider='openai'
# )

# res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c1/dataset_citations.csv')

In [ ]:
# # gpt 5 mini - S3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c2 \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex false

In [ ]:
# batch_id = 'batch_6a46eaad68308190aef947eb95fafbab'

# res = dg.parser.llm_client.download_batch_results(
#     batch_id=batch_id,
#     output_file_path='scripts/tmp/resp_RTR_base.jsonl',
#     api_provider='openai'
# )

# res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c2/dataset_citations.csv')

In [ ]:
# # gpt 5 mini - RS3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c3 \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex true


In [ ]:
# batch_id = 'batch_6a46eb3842d48190aa18ae911c4d31a8'

# res = dg.parser.llm_client.download_batch_results(
#     batch_id=batch_id,
#     output_file_path='scripts/tmp/resp_RTR_base.jsonl',
#     api_provider='openai'
# )

# res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c3/dataset_citations.csv')

In [ ]:
# # gpt-5-mini — FDR
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c4 \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval false --brute-force-regex false \
#     --full-document-read true \
#     --prompt-name GPT_FDR_FewShot

In [ ]:
# batch_id = 'batch_6a4717b720b081909df79f7d3f73a2e9'

# res = dg.parser.llm_client.download_batch_results(
#     batch_id=batch_id,
#     output_file_path='scripts/tmp/resp_RTR_base.jsonl',
#     api_provider='openai'
# )

# res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c4/dataset_citations.csv')

In [ ]:
# # Gemini 3.5 -- base
# !python k8s/k8s_processor.py \
#       --input k8s/input/article_ids_REV_test.csv \
#       --output-dir k8s/output/rev_test_gemini_c1 \
#       --model gemini-3.5-flash \
#       --batch-size 249 \
#       --semantic-retrieval false \
#       --brute-force-regex false \
#       --use-batch-api false

In [ ]:
# # Gemini 3.5 -- S3
# !python k8s/k8s_processor.py \
#       --input k8s/input/article_ids_REV_test.csv \
#       --output-dir k8s/output/rev_test_gemini_c2 \
#       --model gemini-3.5-flash \
#       --batch-size 249 \
#       --semantic-retrieval true \
#       --brute-force-regex false \
#       --use-batch-api false

In [ ]:
# # Gemini 3.5 -- RS3
# !python k8s/k8s_processor.py \
#       --input k8s/input/article_ids_REV_test.csv \
#       --output-dir k8s/output/rev_test_gemini_c3 \
#       --model gemini-3.5-flash \
#       --batch-size 249 \
#       --semantic-retrieval true \
#       --brute-force-regex true \
#       --use-batch-api false

In [ ]:
# # Gemini 3.5 -- FDR
# !python k8s/k8s_processor.py \
# --input k8s/input/article_ids_REV_test.csv \
# --output-dir k8s/output/rev_test_gemini_c4 \
# --model gemini-3.5-flash \
# --batch-size 249 \
# --full-document-read true \
# --use-batch-api false \
# --prompt-name GPT_FDR_FewShot


In [16]:
article_ids_synapse = [re.sub(r'pmc:', '', pmcid) for pmcid in pd.read_csv("scripts/exp_input/syn66046424-20260629.csv")["pmcid"].tolist()]
article_ids_synapse

['PMC10001072',
 'PMC10002464',
 'PMC10005937',
 'PMC10005937',
 'PMC10011140',
 'PMC10011140',
 'PMC10011140',
 'PMC10011140',
 'PMC10011140',
 'PMC10011140',
 'PMC10011140',
 'PMC10011140',
 'PMC10013873',
 'PMC10014304',
 'PMC10025452',
 'PMC10025681',
 'PMC10028786',
 'PMC10028977',
 'PMC10029021',
 'PMC10031009',
 'PMC10039068',
 'PMC10039465',
 'PMC10041447',
 'PMC10041734',
 'PMC10043753',
 'PMC10044680',
 'PMC10044680',
 'PMC10046116',
 'PMC10050155',
 'PMC10050155',
 'PMC10050155',
 'PMC10050155',
 'PMC10054947',
 'PMC10054947',
 'PMC10054947',
 'PMC10054947',
 'PMC10054947',
 'PMC10055057',
 'PMC10055234',
 'PMC10055242',
 'PMC10055503',
 'PMC10055503',
 'PMC10055503',
 'PMC10055654',
 'PMC10055654',
 'PMC10060837',
 'PMC10061294',
 'PMC10061294',
 'PMC10061294',
 'PMC10061294',
 'PMC10066241',
 'PMC10066241',
 'PMC10068328',
 'PMC10074678',
 'PMC10074678',
 'PMC10078980',
 'PMC10080956',
 'PMC10080956',
 'PMC10081415',
 'PMC10081415',
 'PMC10081846',
 'PMC10081846',
 'PMC100

## Results

In [ ]:
! python scripts/BioDMS/eval_configs.py